# Base models on embeddings of `CLS` token

In [1]:
import numpy as np
import pandas as pd

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC, LinearSVC
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

from rital_nlp_project.common.utils import load_clean_data, load_pres
from rital_nlp_project.common.notebook_utils import init_notebook
from rital_nlp_project.common.models.models_eval import eval_pipeline_movies, eval_pipeline_presidents
from rital_nlp_project.movies.models_utils import split_fn

from rital_nlp_project.presidents.models import SmoothedProbaClassifier

init_notebook()

### Bert with mean agg

#### Eval on train/test split

In [3]:
data = pd.read_parquet("../Dataset/embeddings/movies_clean_bert_metadata_bert-base-uncased.parquet")
data_by_example = data.drop_duplicates("example_id")

train_example_ids, test_example_ids, _, _ = split_fn(
    data_by_example["example_id"].values,
    data_by_example["label"].values,
 )

train_mask = data["example_id"].isin(train_example_ids)
X_train = np.stack(data.loc[train_mask, "embedding"].to_numpy())
y_train = data.loc[train_mask, "label"].to_numpy()

test_mask = data["example_id"].isin(test_example_ids)
X_test = np.stack(data.loc[test_mask, "embedding"].to_numpy())
y_test_chunk = data.loc[test_mask, "label"].to_numpy()
test_example_ids_full = data.loc[test_mask, "example_id"].to_numpy()

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

test_proba = model.predict_proba(X_test)[:, 1]

chunk_predictions = pd.DataFrame(
    {
        "example_id": test_example_ids_full,
        "prediction": test_proba,
        "true": y_test_chunk,
    }
)
example_predictions = chunk_predictions.groupby("example_id").agg(
    prediction=("prediction", "mean"),
    true=("true", "first"),
)
example_predictions["pred"] = (example_predictions["prediction"] > 0.5).astype(int)

print(accuracy_score(example_predictions["true"], example_predictions["pred"]))
print(precision_score(example_predictions["true"], example_predictions["pred"]))
print(recall_score(example_predictions["true"], example_predictions["pred"]))
print(f1_score(example_predictions["true"], example_predictions["pred"]))

0.885
0.8532110091743119
0.93
0.8899521531100478


#### Eval on true test

In [ ]:
data_test = pd.read_parquet("../Dataset/embeddings/movies_test_metadata_bert-base-uncased.parquet")
test_labels = pd.read_csv("../Dataset/submissions/submission-movie-1.csv", names=["label"])
data_test = data_test.drop(columns=["label"]).merge(
    test_labels,
    right_index=True,
    left_on="example_id",
)

data_train = pd.read_parquet("../Dataset/embeddings/movies_clean_bert_metadata_bert-base-uncased.parquet")
X_train = np.stack(data_train["embedding"].to_numpy())
y_train = data_train["label"].to_numpy()

X_test = np.stack(data_test["embedding"].to_numpy())

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

test_proba = model.predict_proba(X_test)[:, 1]

data_test["proba_1"] = test_proba
data_test["label"] = data_test["label"].map({"N": 0, "P": 1})
data_test["pred"] = (data_test["proba_1"] > 0.5).astype(int)

example_predictions = data_test.groupby("example_id").agg(
    proba=("proba_1", "mean"),
    label=("label", "first"),
)
example_predictions["pred"] = (example_predictions["proba"] > 0.5).astype(int)

print(accuracy_score(example_predictions["label"], example_predictions["pred"]))
print(precision_score(example_predictions["label"], example_predictions["pred"]))
print(recall_score(example_predictions["label"], example_predictions["pred"]))
print(f1_score(example_predictions["label"], example_predictions["pred"]))

KeyError: "Column(s) ['prediction'] do not exist"

In [26]:
model=LogisticRegression(max_iter=1000)
results = eval_pipeline_movies(model, embeds, classes_movies, cv=True, split=True, clf_report=False)

model=LinearSVC(max_iter=1000)
results = eval_pipeline_movies(model, embeds, classes_movies, cv=True, split=True, clf_report=False)

  model: LogisticRegression(max_iter=1000)
Cross-validation accuracy mean: 0.8900
Cross-validation precision mean: 0.8879
Cross-validation recall mean: 0.8930
Cross-validation f1 mean: 0.8903
Cross-validation roc_auc mean: 0.9561
Training time: 0.135 seconds
Inference time: 0.001 seconds
Test accuracy: 0.8950
Test precision: 0.8989
Test recall: 0.8802
Test f1: 0.8895
Test roc_auc: 0.9566

  model: LinearSVC()
Cross-validation accuracy mean: 0.8875
Cross-validation precision mean: 0.8840
Cross-validation recall mean: 0.8920
Cross-validation f1 mean: 0.8877
Cross-validation roc_auc mean: 0.9531
Training time: 0.203 seconds
Inference time: 0.001 seconds
Test accuracy: 0.8500
Test precision: 0.8424
Test recall: 0.8593
Test f1: 0.8507
Test roc_auc: 0.9416

